# 03 - Model comparison, experiments, error analysis, explainability
Run `python -m src.train` and `python -m src.experiments` once from the project root first, or run the cells below.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, joblib, matplotlib.pyplot as plt, seaborn as sns
from src import config
from src.data import load_raw, prepare
from src.models import split, make_logreg, make_rf, metrics, conf_matrix
from sklearn.metrics import classification_report

## Split (stratified, 80/20) - TF-IDF is fitted inside the pipeline on train only, so no leakage

In [ ]:
df = prepare(load_raw())
X_train, X_test, y_train, y_test = split(df)
print(len(X_train), len(X_test))

## Experiment 2: Logistic Regression vs Random Forest
**Hypothesis:** the linear model matches or beats the forest on sparse TF-IDF text, and is faster.

In [ ]:
rows, fitted = [], {}
for name, pipe in {'Logistic Regression': make_logreg(), 'Random Forest': make_rf()}.items():
    pipe.fit(X_train, y_train); fitted[name] = pipe
    y_pred = pipe.predict(X_test)
    rows.append({'model': name, **metrics(y_test, y_pred)})
    print(name); print(classification_report(y_test, y_pred, target_names=['Genuine','Fake']))
results = pd.DataFrame(rows).round(4); results

In [ ]:
long = results.melt(id_vars='model', var_name='metric', value_name='score')
sns.barplot(data=long, x='metric', y='score', hue='model'); plt.ylim(0, 1.05); plt.show()

## Experiment 1 and 3
Run `python -m src.experiments`, then load the table:

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'experiments.csv')

For each experiment write your own **Hypothesis -> Experiment -> Result -> Analysis -> Conclusion** using the numbers above. Do not copy numbers from anywhere else.

## Error analysis (Experiment 4)

In [ ]:
from src.error_analysis import load_predictions, summarize, examples
ep = load_predictions(); summarize(ep)

In [ ]:
pd.set_option('display.max_colwidth', 200)
examples(ep, 'FP', 10)

In [ ]:
examples(ep, 'FN', 10)

Look for patterns: very short reviews? generic praise? unusual vocabulary? possible label noise?

## Explainability (Logistic Regression)

In [ ]:
from src.explain import global_top_features
fake, genuine = global_top_features(fitted['Logistic Regression'])
print('Fake-leaning words   :', [w for w, _ in fake])
print('Genuine-leaning words:', [w for w, _ in genuine])

Optional SHAP view (needs `pip install shap`):

In [ ]:
try:
    import shap
    pipe = fitted['Logistic Regression']
    Xtr = pipe.named_steps['tfidf'].transform(X_train.sample(1000, random_state=0))
    Xte = pipe.named_steps['tfidf'].transform(X_test.iloc[:5])
    expl = shap.LinearExplainer(pipe.named_steps['clf'], Xtr)
    sv = expl.shap_values(Xte)
    print('SHAP values computed:', sv.shape)
except Exception as e:
    print('SHAP skipped:', e)